# Saved-pair evaluation on Kaggle
Use a Kaggle utility benchmark with model-proxy access; no GPU or training dependencies are needed. This notebook can evaluate newly reproduced pairs when their saved outputs are attached. To inspect the submitted experiment without making model calls, run `python scripts/summarize_saved_evaluation.py` in a local clone. The original judgments are already included; do not rerun them just to verify the report.

In [ ]:
from pathlib import Path
import subprocess,sys,os,json
ROOT=Path('/kaggle/working/cs5326-pa2-small-model-alignment')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/salehfarooq/cs5326-pa2-small-model-alignment.git',str(ROOT)],check=True)
os.chdir(ROOT)
os.environ['PYTHONPATH']=str(ROOT)
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
import kaggle_benchmarks as kbench
from src.judging import run_judging
from src.pairwise import summarize
JUDGE='google/gemini-3.6-flash'


In [ ]:
@kbench.task(name='pa2_saved_pair_judging_reproduction')
def judge_saved(llm):
    names=['pairs-hh-alpaca_eval','pairs-hh-hh','pairs-uf-alpaca_eval','pairs-uf-hh']
    output=Path('/kaggle/working/judged_reproduction');output.mkdir(exist_ok=True)
    def read(p):return [json.loads(line) for line in p.read_text().splitlines()] if p.exists() else []
    comparisons={}
    for name in names:
        files=list(Path('/kaggle/input').rglob(name+'/pairs.jsonl'))
        assert len(files)==1,'Attach the intended generation output exactly once'
        pairs_path=files[0];folder=output/name
        def request(prompt):
            with kbench.chats.new('pa2_anonymous_pairwise_attempt'):return llm.prompt(prompt)
        run_judging(pairs_path,folder,request,JUDGE,batch_size=5,sleep_seconds=5,max_attempts=3,
                    reuse=(folder/'run.json').exists(),bank_kind='hh' if name.endswith('-hh') else 'alpaca_eval')
        comparisons[name]=summarize(read(pairs_path),json.loads((pairs_path.parent/'mapping.json').read_text()),
                   read(folder/'judgments.jsonl'),read(folder/'failures.jsonl'),2000,42)
        (output/'comparisons.json').write_text(json.dumps(comparisons,indent=2)+'\n')
    # The utility task is used for persisted judgments, not a task leaderboard score.
    return None
judge_saved.run(llm=kbench.llms[JUDGE])
